In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

# ==========================================================
# Configuration
# ==========================================================
bronze_table = "bronze_dev.global_mart_retail.raw_data"
silver_table = "silver_dev.global_mart_retail.dim_customer"

# ==========================================================
# 1. Read Bronze Data
# ==========================================================
bronze_df = spark.read.table(bronze_table)

# ==========================================================
# 2. Clean & Standardize Customer Attributes
# ==========================================================
cleaned_df = (
    bronze_df
    .select(
        F.upper(F.trim(F.col("customer_id"))).alias("customer_id"),
        F.lower(F.trim(F.col("customer_name"))).alias("customer_name"),
        F.coalesce(F.lower(F.trim(F.col("segment"))), F.lit("unknown")).alias("customer_segment"),
        F.coalesce(F.lower(F.trim(F.col("country"))), F.lit("unknown")).alias("country"),
        F.coalesce(F.lower(F.trim(F.col("city"))), F.lit("unknown")).alias("city"),
        F.coalesce(F.lower(F.trim(F.col("state"))), F.lit("unknown")).alias("state"),
        F.lpad(
            F.regexp_replace(F.col("postal_code").cast("string"), "[^0-9]", ""),
            5,
            "0"
        ).alias("postal_code"),
        F.coalesce(F.lower(F.trim(F.col("region"))), F.lit("unknown")).alias("region"),
        F.col("ingestion_ts"),
        F.col("row_id")
    )
)

# ==========================================================
# 3. Generate Business Hash
# ==========================================================
hashed_df = (
    cleaned_df
    .withColumn(
        "customer_hash",
        F.sha2(
            F.concat_ws(
                "||",
                "customer_name",
                "customer_segment",
                "country",
                "city",
                "state",
                "postal_code",
                "region"
            ),
            256
        )
    )
)

# ==========================================================
# 4. HARD SOURCE DE-DUPLICATION
#    One row per (customer_id, customer_hash)
# ==========================================================
dedup_window = Window.partitionBy("customer_id", "customer_hash").orderBy(F.col("ingestion_ts").desc())

deduped_df = (
    hashed_df
    .withColumn("rn", F.row_number().over(dedup_window))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

# ==========================================================
# 5. Add SCD2 Metadata and Determine Current Record per Batch
# ==========================================================
current_window = Window.partitionBy("customer_id").orderBy(F.col("ingestion_ts").desc(), F.col("row_id").desc())

staged_df = (
    deduped_df
    .withColumn("rn", F.row_number().over(current_window))
    .withColumn("is_current_record", F.when(F.col("rn") == 1, F.lit(True)).otherwise(F.lit(False)))
    .withColumn("effective_start_timestamp", F.col("ingestion_ts"))
    .withColumn("effective_end_timestamp", F.lit(None).cast("timestamp"))
    .withColumn("load_timestamp", F.current_timestamp())
    .withColumn("batch_id", F.expr("uuid()"))
    .drop("rn")
)

# ==========================================================
# 6. Create Silver Table if Not Exists
# ==========================================================
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {silver_table} (
    customer_key BIGINT GENERATED ALWAYS AS IDENTITY,
    customer_id STRING,
    customer_name STRING,
    customer_segment STRING,
    country STRING,
    city STRING,
    state STRING,
    postal_code STRING,
    region STRING,
    customer_hash STRING,
    effective_start_timestamp TIMESTAMP,
    effective_end_timestamp TIMESTAMP,
    is_current_record BOOLEAN,
    load_timestamp TIMESTAMP,
    batch_id STRING
)
USING DELTA
""")

silver_delta = DeltaTable.forName(spark, silver_table)

# ==========================================================
# 7. MERGE NEW RECORDS (Avoid identity column issues)
# ==========================================================
(
    silver_delta.alias("t")
    .merge(
        staged_df.alias("s"),
        "t.customer_id = s.customer_id AND t.customer_hash = s.customer_hash"
    )
    .whenNotMatchedInsert(
        values={
            "customer_id": "s.customer_id",
            "customer_name": "s.customer_name",
            "customer_segment": "s.customer_segment",
            "country": "s.country",
            "city": "s.city",
            "state": "s.state",
            "postal_code": "s.postal_code",
            "region": "s.region",
            "customer_hash": "s.customer_hash",
            "effective_start_timestamp": "s.effective_start_timestamp",
            "effective_end_timestamp": "s.effective_end_timestamp",
            "is_current_record": "s.is_current_record",
            "load_timestamp": "s.load_timestamp",
            "batch_id": "s.batch_id"
        }
    )
    .execute()
)

# ==========================================================
# 8. COMPUTE EFFECTIVE END DATE USING LEAD
# ==========================================================
silver_df = spark.read.table(silver_table)

# Window to get next record's start timestamp per customer
window_desc = Window.partitionBy("customer_id").orderBy(F.col("effective_start_timestamp").desc())

# Compute new effective_end_timestamp
silver_df = (
    silver_df
    .withColumn(
        "new_effective_end_timestamp",
        F.lag("effective_start_timestamp").over(window_desc)  # previous record's start date
    )
    .withColumn(
        "new_effective_end_timestamp",
        F.when(F.col("is_current_record") == True, None)  # current record remains NULL
         .otherwise(F.col("new_effective_end_timestamp"))
    )
)


# ==========================================================
# 9. UPDATE TABLE WITHOUT TOUCHING IDENTITY COLUMN
# ==========================================================
silver_updates = silver_df.select(
    "customer_key",
    F.col("is_current_record").alias("new_is_current"),
    F.col("new_effective_end_timestamp")
)

(
    silver_delta.alias("t")
    .merge(
        silver_updates.alias("s"),
        "t.customer_key = s.customer_key"
    )
    .whenMatchedUpdate(
        set={
            "is_current_record": "s.new_is_current",
            "effective_end_timestamp": "s.new_effective_end_timestamp"
        }
    )
    .execute()
)

print("✅ SCD Type 2 load completed successfully with correct historical end dates and exactly ONE current record per customer")


In [0]:

# ==========================================================
# 6. Reconciliation & Validation Checks
# ==========================================================
spark.sql(f"""
SELECT
    COUNT(*) AS total_records,
    COUNT(DISTINCT customer_id) AS distinct_customers,
    SUM(CASE WHEN is_current_record THEN 1 ELSE 0 END) AS current_records
FROM {silver_table}
""").show()

In [0]:
spark.sql(f"""
SELECT customer_id, customer_hash, effective_start_timestamp, COUNT(*) AS versions
FROM {silver_table}
GROUP BY customer_id, customer_hash, effective_start_timestamp
HAVING COUNT(*) > 1
ORDER BY versions DESC
""").show()

In [0]:
%sql
select * from silver_dev.global_mart_retail.dim_customer where customer_id = 'SV-20365';